In [ ]:
import pandas as pd
df = pd.read_csv('/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /kanker_nl_master_table.csv')
print(df.shape)        # (73, 7) — sanity check
df.head()              # eyeballing the first 5 rows

(73, 7)


,Cancer Type,Source URL,General Description,Prognosis (Vooruitzichten),Symptoms (Symptomen),Causes (Oorzaken),Treatments (Behandeling)
0,Alvleesklierkanker,https://www.kanker.nl/kankersoorten/alvleeskli...,Alvleesklierkanker is kanker die in de alvlees...,De vooruitzichten bij alvleesklierkanker zijn ...,De meeste mensen met alvleesklierkanker hebben...,De precieze oorzaak van alvleesklierkanker is ...,De meeste mensen met alvleesklierkanker krijge...
1,Anuskanker,https://www.kanker.nl/kankersoorten/anuskanker...,Anuskanker is kanker die ontstaat in de anus. ...,Veel mensen gaan op zoek naar wat anuskanker b...,Bij anuskanker kan zitten of poepen pijnlijk z...,De oorzaak van anuskanker is meestal een infec...,Er zijn verschillende behandelingen mogelijk b...
2,Atypisch fibroxanthoom,https://www.kanker.nl/kankersoorten/atypisch-f...,Een atypisch fibroxanthoomis een zeldzame soor...,Een atypisch fibroxanthoom is goed te behandel...,NaN,De belangrijkste risicofactor voor een atypisc...,Heb je een atypisch fibroxanthoom? Dan krijg j...
3,Baarmoederhalskanker,https://www.kanker.nl/kankersoorten/baarmoeder...,Baarmoederhalskanker is kanker die in de baarm...,Het is begrijpelijk dat je meer wilt weten ove...,Baarmoederhalskanker en de behandeling hebben ...,NaN,Er zijn verschillende behandelingen voor baarm...
4,Baarmoederkanker,https://www.kanker.nl/kankersoorten/baarmoeder...,Baarmoederkanker is kanker die in de baarmoede...,Begrijpelijk dat je meer wilt weten over je vo...,Baarmoederkanker en de behandeling hebben vaak...,Er zijn risicofactoren die de kans op baarmoed...,Er zijn verschillende behandelingen mogelijk b...


In [9]:
def needs_translation(text):
    if pd.isna(text): return False
    if str(text).strip() in ('N/A', '', 'nan'): return False
    return True


In [10]:
def chunk_text(text, max_chars=4500):
    """Split long Dutch text into ≤4,500-char chunks at sentence boundaries."""
    if len(text) <= max_chars:
        return [text]
    sentences = text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
    chunks, current = [], ''
    for s in sentences:
        if len(current) + len(s) <= max_chars:
            current += s + ' '
        else:
            chunks.append(current.strip())
            current = s + ' '
    if current: chunks.append(current.strip())
    return chunks


In [15]:
pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 894.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.1 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
from deep_translator import (
    GoogleTranslator, 
    ChatGptTranslator, 
    MicrosoftTranslator, 
    DeeplTranslator, 
    single_detection, 
    batch_detection
)

In [17]:
import time

translator = GoogleTranslator(source='nl', target='en')

def translate_cell(text):
    if not needs_translation(text):
        return text  # keep "N/A" as "N/A"
    try:
        chunks = chunk_text(str(text))
        english_chunks = [translator.translate(c) for c in chunks]
        time.sleep(0.5)  # be polite to Google's API
        return ' '.join(english_chunks)
    except Exception as e:
        return f"[TRANSLATION_ERROR: {e}]"


In [18]:
columns_to_translate = [
    'General Description',
    'Prognosis (Vooruitzichten)',
    'Symptoms (Symptomen)',
    'Causes (Oorzaken)',
    'Treatments (Behandeling)'
]

for col in columns_to_translate:
    print(f"Translating: {col}")
    df[f'{col}_EN'] = df[col].apply(translate_cell)


Translating: General Description
Translating: Prognosis (Vooruitzichten)
Translating: Symptoms (Symptomen)
Translating: Causes (Oorzaken)
Translating: Treatments (Behandeling)


In [20]:
df.to_csv('/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /kanker_nl_translated_v1.csv', index=False, encoding='utf-8-sig')
df.to_excel('/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /kanker_nl_translated_v1.xlsx', index=False)  # easier for validator
